In [23]:
import json


In [24]:
import json

url = "https://raw.githubusercontent.com/surishettithriloksha-lab/RASM-vul/main/data/primevul_train_paired.jsonl"

with open("/content/train.jsonl", "wb") as f:
    import requests
    f.write(requests.get(url).content)

with open("/content/train.jsonl", "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]

print("Number of training records:", len(train_data))

Number of training records: 7578


In [25]:
valid_url = "https://raw.githubusercontent.com/surishettithriloksha-lab/RASM-vul/main/data/primevul_valid_paired.jsonl"
test_url = "https://raw.githubusercontent.com/surishettithriloksha-lab/RASM-vul/main/data/primevul_test_paired.jsonl"

with open("/content/valid.jsonl", "wb") as f:
    f.write(requests.get(valid_url).content)

with open("/content/test.jsonl", "wb") as f:
    f.write(requests.get(test_url).content)

with open("/content/valid.jsonl", "r", encoding="utf-8") as f:
    valid_data = [json.loads(line) for line in f if line.strip()]

with open("/content/test.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f if line.strip()]

print("Training:", len(train_data))
print("Validation:", len(valid_data))
print("Test:", len(test_data))

Training: 7578
Validation: 960
Test: 870


In [26]:
train_data[0]

{'idx': 0,
 'project': 'openssl',
 'commit_id': 'ca989269a2876bae79393bd54c3e72d49975fc75',
 'project_url': 'https://github.com/openssl/openssl',
 'commit_url': 'https://git.openssl.org/gitweb/?p=openssl.git;a=commit;h=ca989269a2876bae79393bd54c3e72d49975fc75',
 'commit_message': 'Use version in SSL_METHOD not SSL structure.\n\nWhen deciding whether to use TLS 1.2 PRF and record hash algorithms\nuse the version number in the corresponding SSL_METHOD structure\ninstead of the SSL structure. The SSL structure version is sometimes\ninaccurate. Note: OpenSSL 1.0.2 and later effectively do this already.\n(CVE-2013-6449)',
 'target': 1,
 'func': ' long ssl_get_algorithm2(SSL *s)\n        {\n        long alg2 = s->s3->tmp.new_cipher->algorithm2;\n       if (TLS1_get_version(s) >= TLS1_2_VERSION &&\n            alg2 == (SSL_HANDSHAKE_MAC_DEFAULT|TLS1_PRF))\n                return SSL_HANDSHAKE_MAC_SHA256 | TLS1_PRF_SHA256;\n        return alg2;\n\t}\n',
 'func_hash': 25508774765922693275694488

In [27]:
print("Training records:", len(train_data))
print("Validation records:", len(valid_data))
print("Test records:", len(test_data))

print("\nColumns/fields:")
print(train_data[0].keys())

Training records: 7578
Validation records: 960
Test records: 870

Columns/fields:
dict_keys(['idx', 'project', 'commit_id', 'project_url', 'commit_url', 'commit_message', 'target', 'func', 'func_hash', 'file_name', 'file_hash', 'cwe', 'cve', 'cve_desc', 'nvd_url'])


In [28]:
from collections import Counter

print("Training:")
print(Counter(x["target"] for x in train_data))

print("\nValidation:")
print(Counter(x["target"] for x in valid_data))

print("\nTest:")
print(Counter(x["target"] for x in test_data))


Training:
Counter({1: 3789, 0: 3789})

Validation:
Counter({1: 480, 0: 480})

Test:
Counter({1: 435, 0: 435})


In [29]:
# Filter vulnerable (target == 1) records from each split
train_vuln = [x for x in train_data if x["target"] == 1]
valid_vuln = [x for x in valid_data if x["target"] == 1]
test_vuln  = [x for x in test_data  if x["target"] == 1]

print("Vulnerable (target=1) counts:")
print("Train:", len(train_vuln))
print("Valid:", len(valid_vuln))
print("Test:", len(test_vuln))

Vulnerable (target=1) counts:
Train: 3789
Valid: 480
Test: 435


In [30]:
from collections import defaultdict

def build_pairs(data):
    groups = defaultdict(list)
    for record in data:
        key = (record["project"], record["commit_id"])
        groups[key].append(record)

    pairs = []
    for key, records in groups.items():
        vuln = [r for r in records if r["target"] == 1]
        fixed = [r for r in records if r["target"] == 0]
        if len(vuln) == 1 and len(fixed) == 1:
            pairs.append({"vulnerable": vuln[0], "fixed": fixed[0]})
    return pairs

In [31]:
all_pairs = train_pairs + valid_pairs + test_pairs
print("Total pairs:", len(all_pairs))

Total pairs: 4685


In [34]:
from sklearn.model_selection import train_test_split

split_train, split_test = train_test_split(
    all_pairs,
    test_size=0.4,
    random_state=42,
    shuffle=True
)
print("60 40 splitting")
print("60% (train):", len(split_train))
print("40% (test):", len(split_test))

60 40 splitting
60% (train): 2811
40% (test): 1874


In [35]:
from sklearn.model_selection import train_test_split

split_train, split_test = train_test_split(
    all_pairs,
    test_size=0.3,
    random_state=42,
    shuffle=True
)
print("70 30 splitting")
print("70% (train):", len(split_train))
print("30% (test):", len(split_test))

70 30 splitting
70% (train): 3279
30% (test): 1406


In [32]:
from collections import defaultdict

def build_pairs(data):
    groups = defaultdict(list)
    for record in data:
        key = (record["project"], record["commit_id"])
        groups[key].append(record)

    pairs = []
    for key, records in groups.items():
        vuln = [r for r in records if r["target"] == 1]
        fixed = [r for r in records if r["target"] == 0]
        for v, f in zip(vuln, fixed):  # pairs up to min(len(vuln), len(fixed))
            pairs.append({"vulnerable": v, "fixed": f})
    return pairs

train_pairs = build_pairs(train_data)
valid_pairs = build_pairs(valid_data)
test_pairs = build_pairs(test_data)
print("60 20 20 splitting")
print("Train pairs:", len(train_pairs))
print("Valid pairs:", len(valid_pairs))
print("Test pairs:", len(test_pairs))

60 20 20 splitting
Train pairs: 3772
Valid pairs: 480
Test pairs: 433
